In [ ]:
# 设置脚本版本
VER = 6
# 导入所需依赖库
import pandas as pd, numpy as np
# 从指定模块导入工具
from tqdm.notebook import tqdm
# 为 pandas 的 apply 操作启用进度条
tqdm.pandas()
# 导入所需依赖库
import os, sys, pickle, glob, gc
# 从指定模块导入工具
from collections import Counter
# 导入所需依赖库
import cudf, itertools
# 打印运行信息
print('We will use RAPIDS version',cudf.__version__)
# 执行当前语句
pd.set_option('display.max_rows', 500)
# 执行当前语句
pd.set_option('display.max_columns', 500)
# 从指定模块导入工具
from pandarallel import pandarallel
# 初始化 pandarallel 并行环境
pandarallel.initialize(nb_workers=4, progress_bar=True, use_memory_fs=True)
# 导入所需依赖库
import polars as pl
# 从指定模块导入工具
from pyarrow.parquet import ParquetFile
# 导入所需依赖库
import pyarrow as pa 

In [ ]:
# 定义 reduce_memory 函数
def reduce_memory(df):
    # 遍历当前数据
    for col in df.columns:
        # 执行当前语句
        col_type = df[col].dtypes
        # 判断当前条件
        if col_type != object:
            # 执行当前语句
            cmin = df[col].min()
            # 执行当前语句
            cmax = df[col].max()
            # 判断当前条件
            if str(col_type)[:3] == 'int':
                # 判断当前条件
                if cmin > np.iinfo(np.int32).min and cmax < np.iinfo(np.int32).max:
                    # 转换数据类型
                    df[col] = df[col].astype(np.int32)
                # 判断另一个分支条件
                elif cmin > np.iinfo(np.int64).min and cmax < np.iinfo(np.int64).max:
                    # 转换数据类型
                    df[col] = df[col].astype(np.int64)
            # 进入其他情况的处理分支
            else:
                # 判断当前条件
                if cmin > np.finfo(np.float32).min and cmax < np.finfo(np.float32).max:
                    # 转换数据类型
                    df[col] = df[col].astype(np.float32)
                # 进入其他情况的处理分支
                else:
                    # 转换数据类型
                    df[col] = df[col].astype(np.float64)
    # 返回处理结果
    return df

# Feature Extraction

In [ ]:
# 指定当前生成目标
GENERATE_FOR = "kaggle" # "kaggle"

In [ ]:
# 定义行为类型到整数的映射
type_labels = {'clicks':0, 'carts':1, 'orders':2}

In [ ]:
# 设置候选数量
CANDIDATE_COUNT = 100

In [ ]:
# 判断当前条件
if GENERATE_FOR == "local":
    # 设置数据路径
    train_path = "./splitted_raw_data/train.parquet"
    # 设置数据路径
    val_path = "./splitted_raw_data/val.parquet"
# 判断另一个分支条件
elif GENERATE_FOR == "kaggle":
    # 设置数据路径
    train_path = "./splitted_raw_data/all_train.parquet"
    # 设置数据路径
    val_path = "./splitted_raw_data/test.parquet"

### CANDIDATE COVISIT FEATURES

In [ ]:
# 定义 get_covisitation_features 函数
def get_covisitation_features(input_cand_df,
                              # 执行当前语句
                              input_user_int_df,
                              # 执行当前语句
                              input_covisit_df,
                              # 执行当前语句
                              covisit_name="clicks",
                              # 执行当前语句
                              score_col="wgt",
                              # 执行当前语句
                              scoring_name="covisitation",
                              # 执行当前语句
                              session_history_wanted_types=None):      
    # 判断当前条件
    if session_history_wanted_types:
        # 执行当前语句
        filtered_input_user_int_df = input_user_int_df.filter(pl.col('type').is_in(session_history_wanted_types))
    # 进入其他情况的处理分支
    else: 
        # 执行当前语句
        filtered_input_user_int_df = input_user_int_df
    # 执行当前语句
    candidates_w_covisit = input_cand_df[["session", "aid"]].rename({"aid":"aid_x"}).\
    # 执行当前语句
    join(filtered_input_user_int_df.rename({"aid":"aid_y"})[["session", "aid_y"]],
         # 执行当前语句
         how="left",
         # 合并多个数据表
         on="session").fill_null(0).join(input_covisit_df, how="left", on=["aid_x", "aid_y"])
    # 执行当前语句
    candidates_w_covisit = candidates_w_covisit.fill_null(0.)
    # 执行当前语句
    candidates_w_covisit_gby = (
        # 执行当前语句
        candidates_w_covisit
        # 执行当前语句
        .groupby(["session", "aid_x"])
        # 执行当前语句
        .agg(
            # 执行当前语句
            [
                # 执行当前语句
                pl.col(score_col).max().alias(covisit_name + '_' + scoring_name + '_' + "max"),
                # 执行当前语句
                pl.col(score_col).min().alias(covisit_name + '_' + scoring_name + '_' + "min"),
                # 执行当前语句
                pl.col(score_col).std().alias(covisit_name + '_' + scoring_name + '_' + "std"),
                # 执行当前语句
                pl.col(score_col).sum().alias(covisit_name + '_' + scoring_name + '_' + "sum"),
                # 执行当前语句
                pl.col(score_col).mean().alias(covisit_name + '_' + scoring_name + '_' + "mean"),
                # 执行当前语句
                pl.col(score_col).count().alias(covisit_name + '_' + scoring_name + '_' + "count"),
            # 执行当前语句
            ]
        # 执行当前语句
        )
    # 执行当前语句
    ).sort("session", reverse=False)
    # 执行当前语句
    candidates_w_covisit_gby = candidates_w_covisit_gby.rename({"aid_x":"aid"})
    # 转换数据类型
    candidates_w_covisit_gby = candidates_w_covisit_gby.with_column(pl.col("aid").cast(pl.Int32))
    # 转换数据类型
    candidates_w_covisit_gby = candidates_w_covisit_gby.with_column(pl.col("session").cast(pl.Int32)) 
    # 返回处理结果
    return candidates_w_covisit_gby

In [ ]:
# 定义 generate_candidate_history_pair_score_features 函数
def generate_candidate_history_pair_score_features(input_val_df_path,
                                                   # 执行当前语句
                                                   score_df_tuples_w_names,
                                                   # 执行当前语句
                                                   score_col,
                                                   # 执行当前语句
                                                   scoring_name="covisitation"):
    # 读取 parquet 数据
    val_df = pl.read_parquet(input_val_df_path)
    # 遍历当前数据
    for type_str in tqdm(list(type_labels.keys())):
        # 打开 parquet 文件以便分批读取
        pf = ParquetFile(f"./candidate_data/{GENERATE_FOR}_{CANDIDATE_COUNT}candidates_{type_str}.parquet")
        # 执行当前语句
        chunk = 10_000_000
        # 执行当前语句
        total_candidate_df = []
        # 遍历当前数据
        for batch_i, batch in tqdm(enumerate(pf.iter_batches(batch_size = chunk))):
            # 执行当前语句
            candidate_df = batch.to_pandas()
            # 释放临时变量并回收内存
            del batch
            # 执行当前语句
            candidate_df = pl.from_pandas(candidate_df)
            # 转换数据类型
            candidate_df = candidate_df.with_column(pl.col("aid").cast(pl.Int32))
            # 转换数据类型
            candidate_df = candidate_df.with_column(pl.col("session").cast(pl.Int32)) 
            # 转换数据类型
            val_df = val_df.with_column(pl.col("aid").cast(pl.Int32))
            # 转换数据类型
            val_df = val_df.with_column(pl.col("session").cast(pl.Int32))
            # 遍历当前数据
            for covisit in score_df_tuples_w_names:
                # 转换数据类型
                covisit[0] = covisit[0].with_column(pl.col("aid_x").cast(pl.Int32))
                # 转换数据类型
                covisit[0] = covisit[0].with_column(pl.col("aid_y").cast(pl.Int32))
                # 合并多个数据表
                candidate_df = candidate_df.join(
                    # 执行当前语句
                    get_covisitation_features(input_cand_df=candidate_df,
                                              # 执行当前语句
                                              input_user_int_df=val_df,
                                              # 执行当前语句
                                              input_covisit_df=covisit[0],
                                              # 执行当前语句
                                              covisit_name=covisit[1],
                                              # 执行当前语句
                                              score_col=score_col,
                                              # 执行当前语句
                                              scoring_name=scoring_name,
                                              # 执行当前语句
                                              session_history_wanted_types=covisit[2]),
                    # 执行当前语句
                    how="left",
                    # 执行当前语句
                    on=["session", "aid"])
            # 转换数据类型
            candidate_df = candidate_df.with_column(pl.col("aid").cast(pl.Int64))
            # 转换数据类型
            candidate_df = candidate_df.with_column(pl.col("session").cast(pl.Int64))         
            # 执行当前语句
            total_candidate_df.append(candidate_df)
            # 释放临时变量并回收内存
            del candidate_df
        # 拼接多个数据块
        total_candidate_df = pl.concat(total_candidate_df)    
        # 转换数据类型
        total_candidate_df = total_candidate_df.with_columns([pl.col(total_candidate_df.columns[2:]).cast(pl.Float32),])
        # 保存当前结果到 parquet 文件
        total_candidate_df.write_parquet(f'../raw_data/{GENERATE_FOR}_{scoring_name}_features/{scoring_name}_features_{type_str}_{CANDIDATE_COUNT}candidates.pqt')
        # 释放临时变量并回收内存
        del total_candidate_df;gc.collect()

### Covisitation pair 'wgt' features

In [ ]:
# 设置磁盘分片数
DISK_PIECES = 4
# 打印运行信息
print("Reading clicks covisitation...")
# 读取 parquet 数据
clicks_cov_df = pl.from_pandas(pd.concat([pd.read_parquet(f'../raw_data/{GENERATE_FOR}_covisitation/{GENERATE_FOR}_top_{CANDIDATE_COUNT}_clicks_v{VER}_{k}.pqt') for k in range(0, DISK_PIECES)], ignore_index=True))
# 打印运行信息
print("Reading carts-orders covisitation...")
# 读取 parquet 数据
carts_orders_cov_df = pl.from_pandas(pd.concat([pd.read_parquet(f'../raw_data/{GENERATE_FOR}_covisitation/{GENERATE_FOR}_top_{CANDIDATE_COUNT}_carts_orders_v{VER}_{k}.pqt') for k in range(0, DISK_PIECES)], ignore_index=True))
# 打印运行信息
print("Reading buy2buy covisitation...")
# 读取 parquet 数据
buy2buy_cov_df = pl.from_pandas(pd.concat([pd.read_parquet(f'../raw_data/{GENERATE_FOR}_covisitation/{GENERATE_FOR}_top_{CANDIDATE_COUNT}_buy2buy_v{VER}_{k}.pqt') for k in range(0, 1)], ignore_index=True))

In [ ]:
# 定义 all_covisits_features 函数
def all_covisits_features(input_covisit_df,
                         # 执行当前语句
                         score_col,
                         # 执行当前语句
                         covisit_name):
    # 执行当前语句
    aid_cov_feat_df = (
        # 执行当前语句
        clicks_cov_df
        # 执行当前语句
        .groupby(["aid_x"])
        # 执行当前语句
        .agg(
            # 执行当前语句
            [
                # 执行当前语句
                pl.col(score_col).max().alias(covisit_name + '_' + score_col + '_' + "max"),
                # 执行当前语句
                pl.col(score_col).min().alias(covisit_name + '_' + score_col + '_' + "min"),
                # 执行当前语句
                pl.col(score_col).std().alias(covisit_name + '_' + score_col + '_' + "std"),
                # 执行当前语句
                pl.col(score_col).sum().alias(covisit_name + '_' + score_col + '_' + "sum"),
                # 执行当前语句
                pl.col(score_col).mean().alias(covisit_name + '_' + score_col + '_' + "mean"),
                # 执行当前语句
                pl.col(score_col).count().alias(covisit_name + '_' + score_col + '_' + "count"),
            # 执行当前语句
            ]
        # 执行当前语句
        )
    # 执行当前语句
    ).sort("aid_x", reverse=False).rename({"aid_x":"aid"})
    # 转换数据类型
    aid_cov_feat_df = aid_cov_feat_df.with_columns([pl.col(["aid"]).cast(pl.Int64)])
    # 保存当前结果到 parquet 文件
    aid_cov_feat_df.write_parquet(f'../raw_data/{GENERATE_FOR}_covisitation_features/{covisit_name}_covisitation_features.pqt')

In [ ]:
# 组装需要输入的特征表列表
score_df_tuples_w_names = [[clicks_cov_df, "all_clicks"],
                           # 执行当前语句
                           [carts_orders_cov_df, "all_carts_orders"],
                           # 执行当前语句
                           [buy2buy_cov_df, "all_buy2buy"]]
# 遍历当前数据
for covisit_tuple in tqdm(score_df_tuples_w_names):
    # 执行当前语句
    all_covisits_features(input_covisit_df=covisit_tuple[0],
                          # 执行当前语句
                          score_col="wgt",
                          # 执行当前语句
                          covisit_name=covisit_tuple[1])

In [ ]:
# 组装需要输入的特征表列表
score_df_tuples_w_names = [[clicks_cov_df, "clicks", None],
                           # 执行当前语句
                           [carts_orders_cov_df, "carts_orders", None],
                           # 执行当前语句
                           [buy2buy_cov_df, "buy2buy", None]]
# 执行当前语句
generate_candidate_history_pair_score_features(input_val_df_path=val_path,
                                               # 执行当前语句
                                               score_df_tuples_w_names=score_df_tuples_w_names,
                                               # 执行当前语句
                                               score_col="wgt",
                                               # 执行当前语句
                                               scoring_name="covisitation")

### Word2Vec pair 'similarity' features

In [ ]:
# 打印运行信息
print("Reading clicks w2v...")
# 读取 parquet 数据
clicks_cov_df = pl.scan_parquet(f'./all_features/{GENERATE_FOR}_top_{CANDIDATE_COUNT}_clicks_w2v_similarities.pqt')
# 执行当前语句
clicks_cov_df = clicks_cov_df.rename({"similarity":"w2v_similarity"})
# 打印运行信息
print("Reading carts-orders w2v...")
# 读取 parquet 数据
carts_orders_cov_df = pl.scan_parquet(f'./all_features/{GENERATE_FOR}_top_{CANDIDATE_COUNT}_carts_w2v_similarities.pqt')
# 执行当前语句
carts_orders_cov_df = carts_orders_cov_df.rename({"similarity":"w2v_similarity"})
# 打印运行信息
print("Reading buy2buy w2v...")
# 读取 parquet 数据
buy2buy_cov_df = pl.scan_parquet(f'./all_features/{GENERATE_FOR}_top_{CANDIDATE_COUNT}_buy2buy_w2v_similarities.pqt')
# 执行当前语句
buy2buy_cov_df = buy2buy_cov_df.rename({"similarity":"w2v_similarity"})
# 组装需要输入的特征表列表
score_df_tuples_w_names = [[clicks_cov_df, "clicks", None],
                           # 执行当前语句
                           [carts_orders_cov_df, "carts_orders", None],
                           # 执行当前语句
                           [buy2buy_cov_df, "buy2buy", None]]
# 执行当前语句
generate_candidate_history_pair_score_features(input_val_df_path=val_path,
                                               # 执行当前语句
                                               score_df_tuples_w_names=score_df_tuples_w_names,
                                               # 执行当前语句
                                               score_col="w2v_similarity",
                                               # 执行当前语句
                                               scoring_name="word2vec")

## Generating Feature Data Frames

In [ ]:
# 定义 generate_datetime_features 函数
def generate_datetime_features(input_df):
    # 执行当前语句
    input_df["datetime"] = pd.to_datetime(input_df.ts + (2 * 60 * 60), unit='s')
    # 执行当前语句
    input_df["hour"] = input_df["datetime"].dt.hour
    # 执行当前语句
    input_df["dayofweek"] = input_df["datetime"].dt.dayofweek
    # 转换数据类型
    input_df["is_weekend"] = (input_df["dayofweek"]>4).astype(int)
    # 返回处理结果
    return input_df
# 读取 parquet 数据
train_df = pd.read_parquet(train_path)
# 读取 parquet 数据
val_df = pd.read_parquet(val_path)
# 生成当前数据表
train_df = generate_datetime_features(train_df)
# 生成当前数据表
val_df = generate_datetime_features(val_df)
# 拼接多个数据块
item_df = pd.concat([train_df,val_df], ignore_index=True)
# 准备下游特征工程的输入数据
user_df = val_df
# 准备下游特征工程的输入数据
user_item_int_df = val_df    

### Extract AID-Type Occurence Counts & Probabilities

In [ ]:
# 执行当前语句
aid_count_df = item_df[item_df.type==0].groupby("aid")["ts"].count().rename("click_count").to_frame().reset_index()
# 执行当前语句
aid_cart_df = item_df[item_df.type==1].groupby("aid")["ts"].count().rename("cart_count").to_frame().reset_index()
# 执行当前语句
aid_order_df = item_df[item_df.type==2].groupby("aid")["ts"].count().rename("order_count").to_frame().reset_index()
# 合并多个数据表
aid_count_df = aid_count_df.merge(aid_cart_df, how="left", on="aid")
# 合并多个数据表
aid_count_df = aid_count_df.merge(aid_order_df, how="left", on="aid")
# 填充缺失值
aid_count_df.fillna(0., inplace=True)
# 执行当前语句
aid_count_df["click_prob"] = aid_count_df["click_count"] / aid_count_df["click_count"].sum()
# 执行当前语句
aid_count_df["cart_prob"] = aid_count_df["cart_count"] / aid_count_df["cart_count"].sum()
# 执行当前语句
aid_count_df["order_prob"] = aid_count_df["order_count"] / aid_count_df["order_count"].sum()
# 保存当前结果到 parquet 文件
aid_count_df.to_parquet(f'./all_features/{GENERATE_FOR}_aid_occurences.pqt')
# 预览当前结果
display(aid_count_df.head())
# 释放临时变量并回收内存
del aid_count_df, aid_order_df, aid_cart_df; gc.collect()

### Co-Occurence Features

In [ ]:
# 执行当前语句
co_order_v_order = sorted(glob.glob(f"../raw_data/{GENERATE_FOR}_cooccurence/{GENERATE_FOR}_(orders)vs(orders)_cooccurences_v10_*.pqt"))
# 读取 parquet 数据
co_order_v_order = pd.concat([pd.read_parquet(f) for f in co_order_v_order], ignore_index=True)
# 执行当前语句
co_cart_v_cart = sorted(glob.glob(f"../raw_data/{GENERATE_FOR}_cooccurence/{GENERATE_FOR}_(carts)vs(carts)_cooccurences_v10_*.pqt"))
# 读取 parquet 数据
co_cart_v_cart = pd.concat([pd.read_parquet(f) for f in co_cart_v_cart], ignore_index=True)
# 执行当前语句
co_click_v_order = sorted(glob.glob(f"../raw_data/{GENERATE_FOR}_cooccurence/{GENERATE_FOR}_(clicks)vs(orders)_cooccurences_v10_*.pqt"))
# 读取 parquet 数据
co_click_v_order = pd.concat([pd.read_parquet(f) for f in co_click_v_order], ignore_index=True)
# 执行当前语句
co_cart_v_order = sorted(glob.glob(f"../raw_data/{GENERATE_FOR}_cooccurence/{GENERATE_FOR}_(carts)vs(orders)_cooccurences_v10_*.pqt"))
# 读取 parquet 数据
co_cart_v_order = pd.concat([pd.read_parquet(f) for f in co_cart_v_order], ignore_index=True)
# 执行当前语句
co_click_v_cart = sorted(glob.glob(f"../raw_data/{GENERATE_FOR}_cooccurence/{GENERATE_FOR}_(clicks)vs(carts)_cooccurences_v10_*.pqt"))
# 读取 parquet 数据
co_click_v_cart = pd.concat([pd.read_parquet(f) for f in co_click_v_cart], ignore_index=True)
# 执行当前语句
co_order_v_cart = sorted(glob.glob(f"../raw_data/{GENERATE_FOR}_cooccurence/{GENERATE_FOR}_(orders)vs(carts)_cooccurences_v10_*.pqt"))
# 读取 parquet 数据
co_order_v_cart = pd.concat([pd.read_parquet(f) for f in co_order_v_cart], ignore_index=True)

In [ ]:
# 定义 generate_PPMI_features 函数
def generate_PPMI_features(co_df,
                           # 执行当前语句
                           count_df,
                           # 执行当前语句
                           aidx_type="order",
                           # 执行当前语句
                           aidy_type="order"):
    # 执行当前语句
    co_df_w_probs = co_df.\
        # 执行当前语句
        merge(count_df[["aid", f"{aidx_type}_prob"]], left_on="aid_x", right_on="aid").drop("aid", 1).\
                # 执行当前语句
                rename(columns={f"{aidx_type}_prob": "aid_x_prob"}).\
        # 执行当前语句
        merge(count_df[["aid", f"{aidy_type}_prob"]], left_on="aid_y", right_on="aid").drop("aid", 1).\
                # 执行当前语句
                rename(columns={f"{aidy_type}_prob": "aid_y_prob"})
    # 执行当前语句
    co_df_w_probs["pair_prob"] = (co_df_w_probs["wgt"] / co_df_w_probs["wgt"].sum())
    # 执行当前语句
    co_df_w_probs["PPMI"] =\
    # 执行当前语句
    np.maximum(np.log2(co_df_w_probs["pair_prob"] /\
                       # 执行当前语句
                       (co_df_w_probs["aid_x_prob"] * co_df_w_probs["aid_y_prob"])), 0)
    # 返回处理结果
    return co_df_w_probs[["aid_x", "aid_y", "PPMI"]]
# 读取 parquet 数据
aid_count_df = pd.read_parquet(f'./all_features/{GENERATE_FOR}_aid_occurences.pqt')
# 定义需要计算 PPMI 的组合
ppmi_target_combs = [
    # 执行当前语句
    [co_order_v_order, "order", "order"],
    # 执行当前语句
    [co_cart_v_cart, "cart", "cart"],
    # 执行当前语句
    [co_click_v_order, "click", "order"],
    # 执行当前语句
    [co_cart_v_order, "cart", "order"],
    # 执行当前语句
    [co_click_v_cart, "click", "cart"],
    # 执行当前语句
    [co_order_v_cart, "order", "cart"],
# 执行当前语句
]
# 遍历当前数据
for ppmi_target_comb in ppmi_target_combs:
    # 执行当前语句
    ppmi_df = generate_PPMI_features(co_df=ppmi_target_comb[0],
                                     # 执行当前语句
                                     count_df=aid_count_df,
                                     # 执行当前语句
                                     aidx_type=ppmi_target_comb[1],
                                     # 执行当前语句
                                     aidy_type=ppmi_target_comb[2])
    # 保存当前结果到 parquet 文件
    ppmi_df.to_parquet(f'./all_features/{GENERATE_FOR}_ppmi_{ppmi_target_comb[1]}_{ppmi_target_comb[2]}.pqt')
    # 预览当前结果
    display(ppmi_df.head())
    # 释放临时变量并回收内存
    del ppmi_df; gc.collect()
# 释放临时变量并回收内存
del co_order_v_order, co_cart_v_cart, co_click_v_order, co_cart_v_order, co_click_v_cart, co_order_v_cart; gc.collect()

In [ ]:
# 读取 parquet 数据
score_df_tuples_w_names = [[pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_order_cart.pqt"),
                            # 执行当前语句
                            "order_vs_cart", [2]],
                            # 读取 parquet 数据
                            [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_click_cart.pqt"),
                            # 执行当前语句
                            "click_vs_cart", [0]],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_cart_cart.pqt"),
                            # 执行当前语句
                            "cart_vs_cart", [1]],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_click_order.pqt"),
                            # 执行当前语句
                            "click_vs_order", [0]],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_order_order.pqt"),
                            # 执行当前语句
                            "order_vs_order", [2]],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_cart_order.pqt"),
                            # 执行当前语句
                            "cart_vs_order", [1]]
                          # 执行当前语句
                          ]
# 执行当前语句
generate_candidate_history_pair_score_features(input_val_df_path=val_path,
                                               # 执行当前语句
                                               score_df_tuples_w_names=score_df_tuples_w_names,
                                               # 执行当前语句
                                               score_col="PPMI",
                                               # 执行当前语句
                                               scoring_name="ppmi")

In [ ]:
# 读取 parquet 数据
score_df_tuples_w_names = [[pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_order_cart.pqt"),
                            # 执行当前语句
                            "order_vs_cart", None],
                            # 读取 parquet 数据
                            [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_click_cart.pqt"),
                            # 执行当前语句
                            "click_vs_cart", None],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_cart_cart.pqt"),
                            # 执行当前语句
                            "cart_vs_cart", None],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_click_order.pqt"),
                            # 执行当前语句
                            "click_vs_order", None],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_order_order.pqt"),
                            # 执行当前语句
                            "order_vs_order", None],
                           # 读取 parquet 数据
                           [pl.read_parquet(f"./all_features/{GENERATE_FOR}_ppmi_cart_order.pqt"),
                            # 执行当前语句
                            "cart_vs_order", None]
                          # 执行当前语句
                          ]
# 执行当前语句
generate_candidate_history_pair_score_features(input_val_df_path=val_path,
                                               # 执行当前语句
                                               score_df_tuples_w_names=score_df_tuples_w_names,
                                               # 执行当前语句
                                               score_col="PPMI",
                                               # 执行当前语句
                                               scoring_name="ppmi_all_history")

### Common Features

In [ ]:
# 定义 datetime_aggregator 函数
def datetime_aggregator(input_df,
                        # 执行当前语句
                        group_cols=[],
                        # 执行当前语句
                        wanted_cols=[]):
    # 按分组聚合统计特征
    return_df = input_df.groupby(group_cols).agg(
        # 执行当前语句
        {'hour':['mean', 'std'],
         # 执行当前语句
         'dayofweek':['mean', 'std'],
         # 执行当前语句
         'is_weekend':['mean']
        # 执行当前语句
        })
    # 合并多个数据表
    return_df.columns = ['_'.join(group_cols) + '_' +  '_'.join(col) for col in return_df.columns]
    # 返回处理结果
    return return_df
# 定义 type_distribution_aggregator 函数
def type_distribution_aggregator(input_df, 
                                 # 执行当前语句
                                 group_cols=[]):
    # 统计频次并取排序结果
    return_df = input_df.groupby(group_cols)['type'].value_counts(normalize=True)
    # 执行当前语句
    return_df = return_df.unstack('type')
    # 合并多个数据表
    return_df.columns = ['_'.join(group_cols) + '_type' + str(col) + "_mean" for col in return_df.columns]
    # 返回处理结果
    return return_df
# 定义 type_based_aggregator 函数
def type_based_aggregator(input_df,
                          # 执行当前语句
                          group_cols=[],
                          # 执行当前语句
                          wanted_cols=[],
                          # 执行当前语句
                          aggregators=[]):
    # 执行当前语句
    type_dfs = []
    # 遍历当前数据
    for type_id in range(3):
        # 遍历当前数据
        for aggregator in aggregators:
            # 执行当前语句
            aggregator_df = aggregator(input_df[input_df.type==type_id].reset_index(drop=True),
                                       # 执行当前语句
                                       group_cols=group_cols,
                                       # 执行当前语句
                                       wanted_cols=wanted_cols)
            # 执行当前语句
            aggregator_df.columns = ["type" + str(type_id) + "_" + col for col in aggregator_df.columns]
        # 执行当前语句
        type_dfs.append(aggregator_df)
    # 返回处理结果
    return pd.concat(type_dfs, axis=1)
# 定义 existence_amount_aggregator 函数
def existence_amount_aggregator(input_df,
                                 # 执行当前语句
                                 group_cols=[],
                                # 执行当前语句
                                wanted_cols=[],
                                # 执行当前语句
                                return_counts=False):
    # 按分组聚合统计特征
    return_df = input_df.groupby(group_cols).agg({col:["count"] for col in wanted_cols})
    # 合并多个数据表
    return_df.columns = ['_'.join(group_cols) + '_' +  '_'.join(col) for col in return_df.columns]
    # 合并多个数据表
    return_df['_'.join(group_cols) + '_cnt_pct_rank'] =\
            # 转换数据类型
            return_df[return_df.columns[0]].rank(pct=True).astype(np.float32)
    # 执行当前语句
    count_cols = list(return_df.columns[:1])
    # 遍历当前数据
    for count_col in count_cols:  
        # 转换数据类型
        return_df[count_col.replace("count", "existed")] = (return_df[count_col]>0).astype(int)
        # 转换数据类型
        return_df[count_col.replace("count", "existed_multiple")] = (return_df[count_col]>1).astype(int)
        # 判断当前条件
        if return_counts:
            # 转换数据类型
            return_df[count_col.replace("count", "existed_times")] = (return_df[count_col]).astype(int)
    # 执行当前语句
    return_df = return_df[[col for col in return_df.columns if ("count" not in col)]]
    # 返回处理结果
    return return_df
# 定义 nunique_aggregator 函数
def nunique_aggregator(input_df,
                       # 执行当前语句
                       group_cols=[],
                       # 执行当前语句
                       wanted_cols=[]):
    # 按分组聚合统计特征
    return_df = input_df.groupby(group_cols).agg({col:["nunique"] for col in wanted_cols})
    # 合并多个数据表
    return_df.columns = ['_'.join(group_cols) + '_' +  '_'.join(col) for col in return_df.columns]
    # 合并多个数据表
    return_df['_'.join(group_cols) + '_nunq_pct_rank'] =\
            # 转换数据类型
            return_df[return_df.columns[0]].rank(pct=True).astype(np.float32)
    # 返回处理结果
    return return_df

### Item Features

In [ ]:
# 定义 item_interacted_by_sessions_perc 函数
def item_interacted_by_sessions_perc(input_df,
              # 执行当前语句
              group_cols=["aid"],
              # 执行当前语句
              wanted_cols=[]
             # 执行当前语句
             ):
    # 执行当前语句
    return_feature_list = []
    # 执行当前语句
    copy_df = input_df.copy()
    # 执行当前语句
    session_count = copy_df.session.nunique()
    # 执行当前语句
    return_df = 2
    # 遍历当前数据
    for type_str in tqdm(type_labels.keys()):
        # 执行当前语句
        colname = f"session_toitem_{type_str}_perc"
        # 执行当前语句
        type_df = copy_df[copy_df.type==type_labels[type_str]].groupby(["aid", "session"])["ts"].count()    
        # 执行当前语句
        type_df = type_df.reset_index().groupby("aid")["session"].count().rename(colname).to_frame() / session_count
        # 判断当前条件
        if isinstance(return_df, int):
            # 执行当前语句
            return_df = type_df.copy()
        # 进入其他情况的处理分支
        else:
            # 合并多个数据表
            return_df = return_df.merge(type_df, how="outer", on="aid")
    # 返回处理结果
    return return_df.fillna(0.)
# 定义 item_lastweek_features 函数
def item_lastweek_features(input_df,
              # 执行当前语句
              group_cols=["aid"],
              # 执行当前语句
              wanted_cols=[]
             # 执行当前语句
             ):
    # 执行当前语句
    input_df["ts"] = pd.to_datetime(input_df["ts"], unit="s")
    # 执行当前语句
    input_df["week"] = input_df["ts"].dt.week
    # 去重获取唯一值
    all_group_ids = pd.MultiIndex.from_product([input_df.aid.unique(),
                                                # 去重获取唯一值
                                                input_df.week.unique(),
                                                # 执行当前语句
                                                [0,1,2]],
                                               # 执行当前语句
                                               names=['aid', 'week', 'type'])
    # 执行当前语句
    return_dfs = []
    # 遍历当前数据
    for aggfunc in ["count", "nunique"]:
        # 按分组聚合统计特征
        grouped = input_df.groupby(['aid', 'week', 'type'])["session"].agg(aggfunc).rename(aggfunc)
        # 执行当前语句
        grouped = grouped.reindex(all_group_ids, fill_value=0).reset_index()
        # 执行当前语句
        aid_lastweek_occ_ratio = (grouped.groupby(["aid", "type"])[aggfunc].last() /\
                                # 填充缺失值
                                grouped.groupby(["aid", "type"])[aggfunc].sum()).fillna(0.).unstack('type')
        # 执行当前语句
        aid_lastweek_occ_ratio.columns = [ f"type_{col}_lastweek_{aggfunc}_occ_ratio" for col in aid_lastweek_occ_ratio]        
        # 执行当前语句
        return_dfs.append(aid_lastweek_occ_ratio)
        # 填充缺失值
        aid_lastweek_occ_amount = grouped.groupby(["aid", "type"])[aggfunc].last().fillna(0.).unstack('type')
        # 执行当前语句
        aid_lastweek_occ_amount.columns = [ f"type_{col}_lastweek_{aggfunc}_occ_amount" for col in aid_lastweek_occ_amount]        
        # 执行当前语句
        return_dfs.append(aid_lastweek_occ_amount)
        # 执行当前语句
        grouped["pct_change"] = grouped.groupby(["aid", "type"])[aggfunc].pct_change()
        # 填充缺失值
        aid_lastweek_pct_change = grouped.groupby(["aid", "type"])["pct_change"].last().fillna(-999.).\
                    # 执行当前语句
                    replace([np.inf, -np.inf], -999.).unstack('type')
        # 执行当前语句
        aid_lastweek_pct_change.columns = [ f"type_{col}_lastweek_{aggfunc}_pct_change" for col in aid_lastweek_pct_change]
        # 执行当前语句
        return_dfs.append(aid_lastweek_pct_change)
    # 返回处理结果
    return pd.concat(return_dfs, axis=1)
# 定义 item_recurrent_signal 函数
def item_recurrent_signal(input_df,
                          # 执行当前语句
                          group_cols=[],
                          # 执行当前语句
                          wanted_cols=[]):
    # 执行当前语句
    item_recurrent_signal_df = input_df.groupby(['aid','session'])["ts"].nunique()\
            # 执行当前语句
            .rename("recurrent_session_acts_per_item").reset_index()
    # 按分组聚合统计特征
    item_recurrent_signal_df = item_recurrent_signal_df.groupby('aid').agg({
        # 执行当前语句
        'recurrent_session_acts_per_item': ['mean'],
    # 执行当前语句
    })
    # 合并多个数据表
    item_recurrent_signal_df.columns = ['aid_' +  '_'.join(col) for col in item_recurrent_signal_df.columns]
    # 返回处理结果
    return item_recurrent_signal_df

In [ ]:
# 拼接多个数据块
item_features = pd.concat([
    # 执行当前语句
    existence_amount_aggregator(item_df,
                                # 执行当前语句
                                group_cols=["aid"],
                                # 执行当前语句
                                wanted_cols=["session", "aid"]),
    # 执行当前语句
    nunique_aggregator(item_df,
                       # 执行当前语句
                       group_cols=["aid"],
                       # 执行当前语句
                       wanted_cols=["session"]),
    # 执行当前语句
    datetime_aggregator(item_df,
                        # 执行当前语句
                        group_cols=["aid"]),
    # 执行当前语句
    type_distribution_aggregator(item_df,
                                 # 执行当前语句
                                 group_cols=["aid"]),
    # 执行当前语句
    item_interacted_by_sessions_perc(item_df),
    # 执行当前语句
    item_lastweek_features(item_df),
    # 执行当前语句
    item_recurrent_signal(item_df),
    # 执行当前语句
    type_based_aggregator(item_df,
                          # 执行当前语句
                          group_cols=["aid"],
                          # 执行当前语句
                          wanted_cols=["aid", "session"],
                          # 执行当前语句
                          aggregators=[datetime_aggregator,
                                       # 执行当前语句
                                       nunique_aggregator,
                                       # 执行当前语句
                                       existence_amount_aggregator,
                                      # 执行当前语句
                                      item_recurrent_signal])
# 执行当前语句
], axis=1)
# 生成或更新特征表
item_features = reduce_memory(item_features)
# 保存当前结果到 parquet 文件
item_features.to_parquet(f'./all_features/{GENERATE_FOR}_item_features.pqt')
# 打印运行信息
print("Item features are created!")
# 释放临时变量并回收内存
del item_features; gc.collect()

### User Features

In [ ]:
# 定义 session_len 函数
def session_len(input_df,
                # 执行当前语句
                group_cols=["session"],
                # 执行当前语句
                wanted_cols=[],
                # 执行当前语句
                return_min_max=False
               # 执行当前语句
               ):
    # 执行当前语句
    return_df = input_df[group_cols + ["ts"]].copy()
    # 按分组聚合统计特征
    return_df = return_df.groupby(group_cols).agg({"ts":["min", "max"]})
    # 执行当前语句
    return_df.columns = ["session_start", "session_end"]
    # 执行当前语句
    return_df["session_len"] = return_df["session_end"] - return_df["session_start"]
    # 判断当前条件
    if return_min_max:
        # 返回处理结果
        return return_df
    # 进入其他情况的处理分支
    else:
        # 返回处理结果
        return return_df[["session_len"]]
# 定义 partial_session_features 函数
def partial_session_features(input_df,
                        # 执行当前语句
                        group_cols=["session"],
                        # 执行当前语句
                        wanted_cols=[]
           # 执行当前语句
           ):
    # 执行当前语句
    return_feature_list = []
    # 执行当前语句
    return_df = input_df.copy()
    # 转换数据类型
    return_df["ts_diff_gt_thr"] = (return_df.groupby("session")["ts"].diff() > 60*60*6).astype(int)
    # 执行当前语句
    return_df["partial_session_id"] = return_df.groupby("session")["ts_diff_gt_thr"].cumsum()
    # 执行当前语句
    max_partial_sessions_per_session = return_df.groupby("session")["partial_session_id"].\
                # 执行当前语句
                max().rename("max_partial_session_id")
    # 执行当前语句
    return_feature_list.append("max_partial_session_id")
    # 合并多个数据表
    return_df = return_df.merge(max_partial_sessions_per_session, how="left", on="session")
    # 执行当前语句
    mean_nunq_items = return_df.groupby(["session", "partial_session_id"])["aid"].nunique().\
        # 执行当前语句
        reset_index().groupby("session")["aid"].mean().rename("partial_mean_nunique_aid")
    # 合并多个数据表
    return_df = return_df.merge(mean_nunq_items, how="left", on="session")
    # 执行当前语句
    return_feature_list.append("partial_mean_nunique_aid")
    # 执行当前语句
    mean_items = return_df.groupby(["session", "partial_session_id"])["aid"].count().\
        # 执行当前语句
        reset_index().groupby("session")["aid"].mean().rename("partial_mean_count_aid")
    # 合并多个数据表
    return_df = return_df.merge(mean_items, how="left", on="session")
    # 执行当前语句
    return_feature_list.append("partial_mean_count_aid")
    # 遍历当前数据
    for type_str in type_labels.keys():
        # 执行当前语句
        mean_type_colname = f"partial_mean_{type_str}"
        # 执行当前语句
        mean_type = return_df[return_df.type==type_labels[type_str]].groupby(["session", "partial_session_id"])["ts"].count().\
            # 执行当前语句
            reset_index().groupby("session")["ts"].mean().rename(mean_type_colname)
        # 合并多个数据表
        return_df = return_df.merge(mean_type, how="left", on="session").fillna(0)
        # 执行当前语句
        return_feature_list.append(mean_type_colname)    
        # 执行当前语句
        mean_type_colname = f"partial_mean_tsdiff_{type_str}"
        # 执行当前语句
        return_df.loc[return_df.type==type_labels[type_str], "ts_diff"] =\
            # 执行当前语句
            return_df[return_df.type==type_labels[type_str]].groupby(["session", "partial_session_id"])["ts"].\
                    # 执行当前语句
                    diff().rename("ts_diff")
        # 执行当前语句
        ts_meandiff = return_df[return_df.type==type_labels[type_str]].groupby("session")["ts_diff"].mean().rename(mean_type_colname)
        # 合并多个数据表
        return_df = return_df.merge(ts_meandiff, how="left", on="session").fillna(-999)
        # 执行当前语句
        return_feature_list.append(mean_type_colname)    
    # 返回处理结果
    return return_df[["session"] + return_feature_list].groupby("session").first()
# 定义 order_size_stats 函数
def order_size_stats(input_df,
                        # 执行当前语句
                        group_cols=["session"],
                        # 执行当前语句
                        wanted_cols=[]
           # 执行当前语句
           ):
    # 执行当前语句
    return_df = input_df[input_df.type==2].reset_index().copy()
    # 按分组聚合统计特征
    order_size_stat_df = return_df.groupby(["session", "ts"]).agg({"aid":["nunique"]})
    # 执行当前语句
    order_size_stat_df.columns = ["aid_in_orders_nunique"]
    # 按分组聚合统计特征
    order_size_stat_df = order_size_stat_df.reset_index().groupby("session").agg({
        # 执行当前语句
        "aid_in_orders_nunique": ["min", "max", "mean"]
    # 执行当前语句
    })
    # 合并多个数据表
    order_size_stat_df.columns = ['_'.join(group_cols) + '_' +  '_'.join(col) for col in order_size_stat_df.columns]
    # 返回处理结果
    return order_size_stat_df
# 定义 user_action_conversion_ratios 函数
def user_action_conversion_ratios(input_df,
                                  # 执行当前语句
                                  group_cols=["session"],
                                  # 执行当前语句
                                  wanted_cols=[]
                                 # 执行当前语句
                                 ):
    # 执行当前语句
    return_df = input_df.copy()
    # 遍历当前数据
    for i in range(3):
        # 执行当前语句
        return_df.loc[return_df.type==i, f"type_{i}"] = 1
    # 填充缺失值
    return_df.fillna(0., inplace=True)
    # 按分组聚合统计特征
    return_df = return_df.groupby(["session", "aid"]).agg({"type_0": "max",
                                                 # 执行当前语句
                                                 "type_1": "max",
                                                 # 执行当前语句
                                                 "type_2": "max"}).reset_index()
    # 转换数据类型
    return_df["session_click_cart_relation"] = ((return_df["type_0"] == 1.) & return_df["type_1"] == 1.).astype(int)
    # 转换数据类型
    return_df["session_click_order_relation"] = ((return_df["type_0"] == 1.) & return_df["type_2"] == 1.).astype(int)
    # 转换数据类型
    return_df["session_cart_order_relation"] = ((return_df["type_1"] == 1.) & return_df["type_2"] == 1.).astype(int)
    # 按分组聚合统计特征
    return_df = return_df.groupby(["session"]).agg({
        # 执行当前语句
        "session_click_cart_relation":"mean",
        # 执行当前语句
        "session_click_order_relation":"mean",
        # 执行当前语句
        "session_cart_order_relation":"mean",
    # 执行当前语句
    })
    # 返回处理结果
    return return_df

In [ ]:
# 拼接多个数据块
user_features = pd.concat([
    # 执行当前语句
    existence_amount_aggregator(user_df,
                                # 执行当前语句
                                group_cols=["session"],
                                # 执行当前语句
                                wanted_cols=["session", "aid"]),
    # 执行当前语句
    session_len(user_df),
    # 执行当前语句
    nunique_aggregator(user_df,
                       # 执行当前语句
                       group_cols=["session"],
                       # 执行当前语句
                       wanted_cols=["aid"]),
    # 执行当前语句
    datetime_aggregator(user_df,
                        # 执行当前语句
                        group_cols=["session"]),
    # 执行当前语句
    type_distribution_aggregator(user_df,
                                 # 执行当前语句
                                 group_cols=["session"]),
    # 执行当前语句
    partial_session_features(user_df),
    # 执行当前语句
    order_size_stats(user_df),
    # 执行当前语句
    user_action_conversion_ratios(user_df),
    # 执行当前语句
    type_based_aggregator(user_df,
                          # 执行当前语句
                          group_cols=["session"],
                          # 执行当前语句
                          wanted_cols=["aid", "session"],
                          # 执行当前语句
                          aggregators=[datetime_aggregator,
                                       # 执行当前语句
                                       session_len,
                                       # 执行当前语句
                                       nunique_aggregator,
                                       # 执行当前语句
                                       existence_amount_aggregator])
# 执行当前语句
], axis=1)
# 生成或更新特征表
user_features = reduce_memory(user_features)
# 保存当前结果到 parquet 文件
user_features.to_parquet(f'./all_features/{GENERATE_FOR}_user_features.pqt')
# 打印运行信息
print("User features are created!")

In [ ]:
# 释放临时变量并回收内存
del user_features

### User-Item Interaction Features

In [ ]:
# 定义 is_last_aid_of_the_session 函数
def is_last_aid_of_the_session(input_df,
                               # 执行当前语句
                               group_cols=["session", "aid"],
                               # 执行当前语句
                               wanted_cols=[]
                              # 执行当前语句
                              ):
    # 执行当前语句
    return_df = input_df[group_cols].copy()
    # 执行当前语句
    return_df["is_aid_interacted_last"] = 0
    # 执行当前语句
    return_df.loc[return_df.session.shift(-1) != return_df.session, "is_aid_interacted_last"] = 1
    # 按分组聚合统计特征
    return_df = return_df.groupby(group_cols).agg({"is_aid_interacted_last":["max"]})
    # 执行当前语句
    return_df.columns = ["is_aid_interacted_last_in_session"]
    # 返回处理结果
    return return_df
# 定义 aid_session_ts_offsets 函数
def aid_session_ts_offsets(input_df,
                # 执行当前语句
                group_cols=["session", "aid"],
                # 执行当前语句
                wanted_cols=[]):
    # 执行当前语句
    session_lens = session_len(input_df,
                               # 执行当前语句
                               return_min_max=True).reset_index()
    # 执行当前语句
    return_df = input_df[group_cols + ["ts"]].copy()
    # 按分组聚合统计特征
    return_df = return_df.groupby(group_cols).agg({"ts":["last"]})
    # 执行当前语句
    return_df.columns = ["session_aid_last_ts"]
    # 执行当前语句
    return_df.reset_index(inplace=True)
    # 合并多个数据表
    return_df = return_df.merge(session_lens, how="left", on="session")
    # 执行当前语句
    return_df["aid_ts_session_end_offset"] = return_df["session_end"] - return_df["session_aid_last_ts"]
    # 执行当前语句
    return_df["aid_ts_session_start_offset"] = return_df["session_aid_last_ts"] - return_df["session_start"]
    # 执行当前语句
    return_df = return_df[group_cols + ["aid_ts_session_end_offset", "aid_ts_session_start_offset"]].set_index(group_cols)
    # 返回处理结果
    return return_df
# 定义 reverse_order_of_aid_for_session 函数
def reverse_order_of_aid_for_session(input_df,
                               # 执行当前语句
                               group_cols=["session", "aid"],
                               # 执行当前语句
                               wanted_cols=[]
                              # 执行当前语句
                              ):
    # 执行当前语句
    return_df = input_df[group_cols].copy()
    # 执行当前语句
    return_df.loc[:, "session_action_order"] = return_df.groupby("session")["aid"].cumcount()
    # 执行当前语句
    session_aid_counts = return_df.groupby("session")["aid"].count()\
                        # 执行当前语句
                        .rename("session_action_count").reset_index()
    # 合并多个数据表
    return_df = return_df.merge(session_aid_counts, how="left", on="session")
    # 执行当前语句
    return_df["session_action_order"] = return_df["session_action_count"] - return_df["session_action_order"]
    # 执行当前语句
    return_df = return_df.groupby(group_cols)["session_action_order"].min().rename("session_action_last_order").to_frame()
    # 返回处理结果
    return return_df

In [ ]:
# 拼接多个数据块
user_item_int_features = pd.concat([
    # 执行当前语句
    existence_amount_aggregator(user_item_int_df,
                                # 执行当前语句
                                group_cols=["session", "aid"],
                                # 执行当前语句
                                wanted_cols=["aid"],
                                # 执行当前语句
                                return_counts=True),
    # 执行当前语句
    reverse_order_of_aid_for_session(user_item_int_df),
    # 执行当前语句
    aid_session_ts_offsets(user_item_int_df),
    # 执行当前语句
    datetime_aggregator(user_item_int_df,
                        # 执行当前语句
                        group_cols=['session', 'aid']),
    # 执行当前语句
    type_distribution_aggregator(user_item_int_df,
                                 # 执行当前语句
                                 group_cols=['session', 'aid']),
    # 执行当前语句
    type_based_aggregator(user_item_int_df,
                          # 执行当前语句
                          group_cols=['session', 'aid'],
                          # 执行当前语句
                          wanted_cols=["aid"],
                          # 执行当前语句
                          aggregators=[datetime_aggregator,
                                       # 执行当前语句
                                       reverse_order_of_aid_for_session,
                                       # 执行当前语句
                                       aid_session_ts_offsets,
                                       # 执行当前语句
                                       existence_amount_aggregator])
# 执行当前语句
], axis=1)
# 生成或更新特征表
user_item_int_features = reduce_memory(user_item_int_features)
# 保存当前结果到 parquet 文件
user_item_int_features.to_parquet(f'./all_features/{GENERATE_FOR}_user_item_int_features.pqt')
# 打印运行信息
print("User-Item Interaction features are created!")

In [ ]:
# 释放临时变量并回收内存
del user_item_int_features
# 手动执行垃圾回收
gc.collect()

In [ ]:
# 释放临时变量并回收内存
del item_df, train_df

## Merging Features w/ Candidates

In [ ]:
# 打印运行信息
print("Reading item features...")
# 读取 parquet 数据
item_features = pl.scan_parquet(f'./all_features/{GENERATE_FOR}_item_features.pqt')
# 打印运行信息
print("Reading user features...")
# 读取 parquet 数据
user_features = pl.scan_parquet(f'./all_features/{GENERATE_FOR}_user_features.pqt')
# 打印运行信息
print("Reading user-item interaction features...")
# 读取 parquet 数据
user_item_int_features = pl.scan_parquet(f'./all_features/{GENERATE_FOR}_user_item_int_features.pqt')
# 读取 parquet 数据
val_df = pl.scan_parquet(val_path)
# 遍历当前数据
for type_str in tqdm(list(type_labels.keys())):
    # 读取 parquet 数据
    covisit_feature_df = pl.scan_parquet(f'../raw_data/{GENERATE_FOR}_covisitation_features/covisitation_features_{type_str}_{CANDIDATE_COUNT}candidates.pqt')
    # 读取 parquet 数据
    all_clicks_covisit_feature_df = pl.scan_parquet(f'../raw_data/{GENERATE_FOR}_covisitation_features/all_clicks_covisitation_features.pqt')
    # 读取 parquet 数据
    all_cart_covisit_feature_df = pl.scan_parquet(f'../raw_data/{GENERATE_FOR}_covisitation_features/all_carts_orders_covisitation_features.pqt')
    # 读取 parquet 数据
    all_buy2buy_covisit_feature_df = pl.scan_parquet(f'../raw_data/{GENERATE_FOR}_covisitation_features/all_buy2buy_covisitation_features.pqt')
    # 读取 parquet 数据
    ppmi_all_feature_df = pl.scan_parquet(f'../raw_data/{GENERATE_FOR}_ppmi_all_history_features/ppmi_all_history_features_{type_str}_{CANDIDATE_COUNT}candidates.pqt')
    # 打开 parquet 文件以便分批读取
    pf = ParquetFile(f"./candidate_data/{GENERATE_FOR}_{CANDIDATE_COUNT}candidates_{type_str}.parquet")
    # 执行当前语句
    chunk = 10_000_000
    # 执行当前语句
    total_candidate_df = 0
    # 遍历当前数据
    for batch_i, batch in tqdm(enumerate(pf.iter_batches(batch_size = chunk))):
        # 执行当前语句
        candidate_df = batch.to_pandas()
        # 执行当前语句
        candidate_df = pl.from_pandas(candidate_df)  
        # 生成候选的排名序列
        rank_repeater = np.hstack([list(range(1,CANDIDATE_COUNT+1)) for i in range(int(len(candidate_df)/CANDIDATE_COUNT))])
        # 执行当前语句
        candidate_df = candidate_df.with_column(pl.Series(name="candidate_rank", values=rank_repeater))
        # 释放临时变量并回收内存
        del rank_repeater;gc.collect()
        # 合并多个数据表
        candidate_df = candidate_df.join(covisit_feature_df, on=['session',
                                                                 # 执行当前语句
                                                                 'aid'], how='left').fill_null(-1)
        # 去重获取唯一值
        candidate_df = candidate_df.unique()
        # 合并多个数据表
        candidate_df = candidate_df.join(ppmi_all_feature_df, on=['session',
                                                              # 执行当前语句
                                                              'aid'], how='left').fill_null(-1)
        # 去重获取唯一值
        candidate_df = candidate_df.unique()
        # 合并多个数据表
        candidate_df = candidate_df.join(all_clicks_covisit_feature_df, on=['aid'], how='left').fill_null(-1)
        # 合并多个数据表
        candidate_df = candidate_df.join(all_cart_covisit_feature_df, on=['aid'], how='left').fill_null(-1)
        # 合并多个数据表
        candidate_df = candidate_df.join(all_buy2buy_covisit_feature_df, on=['aid'], how='left').fill_null(-1)
        # 去重获取唯一值
        candidate_df = candidate_df.unique()
        # 合并多个数据表
        candidate_df = candidate_df.join(item_features, on='aid', how='left').fill_null(-1)
        # 合并多个数据表
        candidate_df = candidate_df.join(user_features, on='session', how='left').fill_null(-1)
        # 合并多个数据表
        candidate_df = candidate_df.join(user_item_int_features,
                                          # 执行当前语句
                                          on=['session', 'aid'],
                                          # 执行当前语句
                                          how='left').fill_null(-1)
        # 读取 parquet 数据
        tar = pd.read_parquet('./splitted_raw_data/val_labels.parquet')
        # 处理标签数据
        tar = tar.loc[ tar['type'] == type_str ]
        # 展开列表列为逐行数据
        aids = tar.ground_truth.explode().rename('aid')
        # 处理标签数据
        tar = tar[['session']]
        # 合并多个数据表
        tar = tar.merge(aids, left_index=True, right_index=True, how='left')
        # 执行当前语句
        tar['label'] = 1
        # 处理标签数据
        tar = pl.from_pandas(tar)
        # 合并多个数据表
        candidate_df = candidate_df.join(tar, on=['session','aid'], how='left').fill_null(0)
        # 去重获取唯一值
        candidate_df = candidate_df.unique()
        # 保存当前结果到 parquet 文件
        candidate_df.write_parquet(f'./candidated_features/{GENERATE_FOR}_{type_str}_all_data_{CANDIDATE_COUNT}candidates_p{batch_i}.pqt')
        # 释放临时变量并回收内存
        del candidate_df,tar,aids;gc.collect()
    # 释放临时变量并回收内存
    del covisit_feature_df;gc.collect()